# **Task 04: Graph querying**

In [ ]:
!pip install rdflib
github_storage = "https://raw.githubusercontent.com/FacultadInformatica-LinkedData/Curso2026-2027/refs/heads/master/Assignment4/course_materials"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 15.8 MB/s eta 0:00:00


In [ ]:
from rdflib import Graph, Namespace, Literal
g = Graph()
g.parse(github_storage+"/rdf/example3.rdf", format="xml")

from rdflib.plugins.sparql import prepareQuery
from rdflib import XSD
from rdflib import FOAF

VCARD = Namespace("http://www.w3.org/2001/vcard-rdf/3.0#")

Spanish: Listar todos los recursos que contienen la propiedad VCARD:FN

English: List all resources that contain the VCARD:FN property

In [ ]:
q = prepareQuery('''
  SELECT ?Subject WHERE {
    ?Subject vcard:FN ?FullName.
  }
  ''',
  initNs = { "vcard": VCARD}
)

for r in g.query(q):
  print(r.Subject)

http://somewhere#JaneSmith
http://somewhere#JohnSmith
http://somewhere#SarahJones
http://somewhere#MattJones


Spanish Repetir la anterior consulta, pero pidiendo ahora además los nombres completos de los sujetos

English: Repeat the previous query, but this time also request the full names of the subjects

In [ ]:
q = prepareQuery('''
  SELECT ?Subject ?FullName WHERE {
    ?Subject vcard:FN ?FullName.
  }
  ''',
  initNs = { "vcard": VCARD}
)

for r in g.query(q):
  print(r.Subject, r.FullName)

http://somewhere#JaneSmith Jane Smith
http://somewhere#JohnSmith John Smith
http://somewhere#SarahJones Sarah Jones
http://somewhere#MattJones Matt Jones


Spanish: Obtener todos los recursos que contienen "Smith" como nombre de familia

English: Retrieve all resources containing ‘Smith’ as a surname

In [ ]:
q = prepareQuery('''
  SELECT ?Subject WHERE {
    ?Subject vcard:Family ?Family.
  }
  ''',
  initNs = { "vcard": VCARD}
)

for r in g.query(q, initBindings = {'?Family' : Literal('Smith', datatype=XSD.string)}):
  print(r.Subject)

print("-----")

q = prepareQuery('''
  SELECT ?Subject WHERE {
    ?Subject vcard:Family "Smith"^^xsd:string.
  }
  ''',
  initNs = { "vcard": VCARD }
)

for r in g.query(q):
  print(r.Subject)

http://somewhere#JaneSmith
http://somewhere#JohnSmith
-----
http://somewhere#JaneSmith
http://somewhere#JohnSmith


Spanish: Obtener todos los elementos que contienen un email asociado

English: Retrieve all items that have an associated email address



In [ ]:
q = prepareQuery('''
  SELECT ?Subject ?Email WHERE {
    ?Subject foaf:email ?Email.
  }
  ''',
  initNs = { "foaf": FOAF}
)

for r in g.query(q):
  print(r.Subject,r.Email)

http://somewhere#JaneSmith jSmith@somewhere.com
http://somewhere#SarahJones sJones@somewhere.com


Spanish: Consultar todos los que conocen (FOAF:knows) a "Jane Smith" y obtenemos sus nombres de pila (VCARD:Given)

English: Query everyone who knows (FOAF:knows) “Jane Smith” and retrieve their first names (VCARD:Given)

In [ ]:
q = prepareQuery('''
  SELECT  ?Subject ?Given  WHERE {
    ?Subject foaf:knows ?JaneSmith.
	?JaneSmith vcard:FN ?JaneSmithFullName.
	?Subject vcard:Given ?Given.
  }
  ''',
  initNs = { "foaf": FOAF, "vcard": VCARD, "xsd":XSD}
)

for r in g.query(q, initBindings = {'?JaneSmithFullName' : Literal('Jane Smith', datatype=XSD.string)}):
  print(r.Subject, r.Given)

http://somewhere#JohnSmith John
http://somewhere#MattJones Matt


# **Task 04: Do the following exercises**

Spanish: Listar el nombre completo y el email de quienes tengan ambos

English: List the full name and email of those who have both

In [ ]:
#TODO
q = prepareQuery('''
  SELECT ?FullName ?Email WHERE {
    ?Subject vcard:FN ?FullName.
    ?Subject foaf:email ?Email.
  }
  ''',
  initNs = { "vcard": VCARD, "foaf": FOAF }
)
for r in g.query(q):
  print(f"{r.FullName} tiene la siguiente dirección email: {r.Email}")

Jane Smith tiene la siguiente dirección email: jSmith@somewhere.com
Sarah Jones tiene la siguiente dirección email: sJones@somewhere.com


Spanish: Listar todos los nombres completos ordenados alfabéticamente (ORDER BY)

English: List all full names in alphabetical order (ORDER BY)

In [ ]:
#TODO
q = prepareQuery('''
  SELECT ?FullName WHERE {
    ?Subject vcard:FN ?FullName.
  } ORDER BY ?FullName
  ''',
  initNs = { "vcard": VCARD }
)
for r in g.query(q):
  print(r.FullName)

Jane Smith
John Smith
Matt Jones
Sarah Jones


Spanish: Listar el nombre completo de quienes no tienen email

English: List the fullname of those who do not have an email

Pista/Hint: Use "FILTER NOT EXISTS { ?Subject foaf:email ?Email. }"

In [ ]:
#TODO
q = prepareQuery('''
  SELECT ?FullName WHERE {
    ?Subject vcard:FN ?FullName.
    FILTER NOT EXISTS { ?Subject foaf:email ?Email. }
  }
  ''',
  initNs = { "vcard": VCARD, "foaf": FOAF }
)
for r in g.query(q):
  print(r.FullName)

John Smith
Matt Jones


Spanish: Comprobar si existe algún recurso con el apellido "García"

English: Check whether any resource has the surname "García"

In [ ]:
#TODO
q = prepareQuery('''
  ASK {
    ?Subject vcard:Family ?Surname.
  }
  ''',
  initNs = { "vcard": VCARD, "xsd": XSD }
)
print(bool(g.query(q, initBindings = {'?Surname' : Literal('García', datatype=XSD.string)})))

print("----")

q = prepareQuery('''
  ASK {
    ?Subject vcard:Family "García"^^xsd:string.
  }
  ''',
  initNs = { "vcard": VCARD, "xsd": XSD }
)
print(bool(g.query(q)))

False
----
False


Spanish: Obtener el nombre completo de las personas con apellido "Smith" a las que conoce Matt Jones usando initBinding.

English: Retrieve the full names of the people with surname "Smith" whom Matt Jones knows using initBindings.

In [ ]:
from rdflib import URIRef

q = prepareQuery('''
  SELECT ?FriendName WHERE {
    ?Person foaf:knows ?Friend.
    ?Friend vcard:Family ?Family.
    ?Friend vcard:FN ?FriendName.
  }
  ''',
  initNs = { "vcard": VCARD, "foaf": FOAF }
)

for r in g.query(q, initBindings = {'?Person' : URIRef('http://somewhere#MattJones'), '?Family' : Literal('Smith', datatype=XSD.string)}):
  print(f"Matt Jones conoce a {r.FriendName}")

print("----")

q = prepareQuery('''
  SELECT ?FriendName WHERE {
    <http://somewhere#MattJones> foaf:knows ?Friend.
    ?Friend vcard:Family "Smith"^^xsd:string.
    ?Friend vcard:FN ?FriendName.
  }
  ''',
  initNs = { "vcard": VCARD, "foaf": FOAF }
)

for r in g.query(q):
  print(f"Matt Jones conoce a {r.FriendName}")

Matt Jones conoce a Jane Smith
----
Matt Jones conoce a Jane Smith
